# 学習履歴の可視化

In [ ]:
!pip install emoji

In [ ]:
import html
import emoji

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
import keras
import keras_nlp
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, f1_score
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Input data files are available in the read-only "../input/" directory
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Make sure the competition data files are found
competition_path = '/kaggle/input/nlp-getting-started'
train_path = os.path.join(competition_path, 'train.csv')
test_path = os.path.join(competition_path, 'test.csv')
submission_path = os.path.join(competition_path, 'sample_submission.csv')

print(f"\nChecking if competition files exist:")
print(f"Train file exists: {os.path.exists(train_path)}")
print(f"Test file exists: {os.path.exists(test_path)}")
print(f"Sample submission exists: {os.path.exists(submission_path)}")


import re
import string
import os

print("TensorFlow version:", tf.__version__)
print("KerasNLP version:", keras_nlp.__version__)

In [ ]:
import re

STOPWORDS = set([
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', 
    'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 
    'she', 'her', 'hers', 'herself', 'it', 'its', 'itself', 'they', 'them', 
    'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 
    'that', 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 
    'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 
    'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 
    'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 
    'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 
    'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 
    'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 
    'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 
    'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very', 
    's', 't', 'can', 'will', 'just', 'don', 'should', 'now'
])

CONTRACTION_MAP = {
    "can't": "cannot", "I'm": "I am", "he's": "he is", "you're": "you are", "it's": "it is",
    "won't": "will not", "they've": "they have", "we're": "we are", "didn't": "did not",
    "hasn't": "has not", "haven't": "have not", "I'd": "I would", "we'll": "we will"
}

def expand_contractions(text, contraction_map=CONTRACTION_MAP):
    contractions_pattern = re.compile('({})'.format('|'.join(re.escape(k) for k in contraction_map.keys())),
                                      flags=re.IGNORECASE | re.DOTALL)

    def expand_match(contraction):
        match = contraction.group(0)
        lower_match = match.lower()
        expanded_contraction = contraction_map.get(match) or contraction_map.get(lower_match)
        if expanded_contraction:
            if match[0].isupper():
                return expanded_contraction[0].upper() + expanded_contraction[1:]
            else:
                return expanded_contraction
        else:
            return match

    return contractions_pattern.sub(expand_match, text)

# テスト例
text = "I'm sure he won't do it because he's lazy."
expanded = expand_contractions(text)
print(expanded)


In [ ]:
import re
import html
import emoji
import string

# === 1. 略語辞書と関数 ===
CONTRACTION_MAP = {
    "can't": "cannot", "I'm": "I am", "he's": "he is", "you're": "you are", "it's": "it is",
    "won't": "will not", "they've": "they have", "we're": "we are", "didn't": "did not",
    "hasn't": "has not", "haven't": "have not", "I'd": "I would", "we'll": "we will"
    # 必要に応じて追加してOK
}

def expand_contractions(text, contraction_map=CONTRACTION_MAP):
    pattern = re.compile('({})'.format('|'.join(re.escape(k) for k in contraction_map.keys())),
                         flags=re.IGNORECASE | re.DOTALL)
    def expand_match(match):
        found = match.group(0)
        lower_found = found.lower()
        expanded = contraction_map.get(found) or contraction_map.get(lower_found)
        if expanded:
            return expanded[0].upper() + expanded[1:] if found[0].isupper() else expanded
        return found
    return pattern.sub(expand_match, text)

# === 2. BERT向けクリーニング関数 ===
def clean_text_for_bert(text):
    if not isinstance(text, str):
        return ''
    text = html.unescape(text)
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = expand_contractions(text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^A-Za-z0-9\s.,!?\'`]', '', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# === 3. 通常クリーニング関数 ===
def clean_text_traditional(text):
    if isinstance(text, str):
        text = text.lower()
        text = re.sub(r'http\S+|www\S+|https\S+', '', text)
        text = re.sub(r'@\w+|#\w+', '', text)
        text = text.translate(str.maketrans('', '', string.punctuation))
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    else:
        return ''


In [ ]:
df_train = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
df_test = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

print('Training Set Shape = {}'.format(df_train.shape))
print('Training Set Memory Usage = {:.2f} MB'.format(df_train.memory_usage().sum() / 1024**2))
print('Test Set Shape = {}'.format(df_test.shape))
print('Test Set Memory Usage = {:.2f} MB'.format(df_test.memory_usage().sum() / 1024**2))

# データ読み込み＆基本情報確認

In [ ]:
import pandas as pd

df_train = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
df_test = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

print('Training Set Shape = {}'.format(df_train.shape))
print('Training Set Memory Usage = {:.2f} MB'.format(df_train.memory_usage().sum() / 1024**2))
print('Test Set Shape = {}'.format(df_test.shape))
print('Test Set Memory Usage = {:.2f} MB'.format(df_test.memory_usage().sum() / 1024**2))


# EDA（探索的データ解析）と欠損値処理

In [ ]:
print("\n=== 探索的データ解析 ===")
print("Training set columns:", df_train.columns.tolist())
print("\nMissing values in training set:")
print(df_train.isnull().sum())
print("\nTarget distribution:")
print(df_train['target'].value_counts())
print(df_train['target'].value_counts(normalize=True).round(3))

# 欠損値を 'unknown' に置換
df_train['location'].fillna('unknown', inplace=True)
df_train['keyword'].fillna('unknown', inplace=True)
df_test['location'].fillna('unknown', inplace=True)
df_test['keyword'].fillna('unknown', inplace=True)


# テキストクリーニングの適用

In [ ]:
# 事前に clean_text_for_bert 関数が定義されている必要あり
print("\n=== テキストクリーニング ===")
print("原文例:")
print(df_train['text'].iloc[0])

df_train['clean_text'] = df_train['text'].apply(clean_text_for_bert)
df_test['clean_text'] = df_test['text'].apply(clean_text_for_bert)

print("\nテキストクリーニング例:")
print(df_train['clean_text'].iloc[0])


# テキスト長の分析

In [ ]:
df_train["length"] = df_train["clean_text"].apply(lambda x : len(x))
df_test["length"] = df_test["clean_text"].apply(lambda x : len(x))

print("\n=== テキスト長 ===")
print("Train Length Stat (cleaned text)")
print(df_train["length"].describe())
print()
print("Test Length Stat (cleaned text)")
print(df_test["length"].describe())


# 単語数のカウントと可視化

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_train['word_count'] = df_train['clean_text'].apply(lambda x: len(x.split()))

plt.figure(figsize=(10, 6))
sns.histplot(data=df_train, x='word_count', hue='target', bins=30, kde=True)
plt.title('Word Count Distribution by Target (After Cleaning)')
plt.xlabel('Word Count')
plt.ylabel('Frequency')
plt.legend(['Not Disaster', 'Disaster'])
plt.show()


# モデル用パラメータ設定

In [ ]:
BATCH_SIZE = 32
NUM_TRAINING_EXAMPLES = df_train.shape[0]
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.2
STEPS_PER_EPOCH = int(NUM_TRAINING_EXAMPLES)*TRAIN_SPLIT // BATCH_SIZE

EPOCHS = 20  
AUTO = tf.data.experimental.AUTOTUNE


#  データ分割とモデル準備

In [ ]:
from sklearn.model_selection import train_test_split
import keras_nlp
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

X = df_train["clean_text"]
y = df_train["target"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=VAL_SPLIT, random_state=42)
X_test = df_test["clean_text"]

preset= "distil_bert_base_en_uncased"

preprocessor = keras_nlp.models.DistilBertPreprocessor.from_preset(preset,
                                                                   sequence_length=160,
                                                                   name="preprocessor_4_tweets"
                                                                  )

classifier = keras_nlp.models.DistilBertClassifier.from_preset(preset,
                                                               preprocessor = preprocessor, 
                                                               num_classes=2)

classifier.summary()

classifier.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    metrics= ["accuracy"]  
)

early_stopping = EarlyStopping(monitor='val_loss', 
                               patience=3, 
                               restore_best_weights=True,
                               verbose=1)

reduce_lr = ReduceLROnPlateau(monitor='val_loss', 
                              factor=0.5, 
                              patience=2, 
                              min_lr=1e-7,
                              verbose=1)

# モデル学習

In [ ]:
EPOCHS = 3
BATCH_SIZE = 64

X_train_small = X_train.sample(frac=0.3, random_state=42)
y_train_small = y_train.loc[X_train_small.index]

history = classifier.fit(
    x=X_train_small,
    y=y_train_small,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    verbose=1,
    callbacks=[early_stopping, reduce_lr]
)


# 学習履歴の可視化

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()


# モデル保存

In [ ]:
print("\n=== モデルを保存中... ===")
classifier.save("distilbert_classifier.keras") 
print("モデルはファイルに正常に保存されました 'distilbert_classifier.keras'")


# モデル評価（混同行列）

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np

def displayConfusionMatrix(y_true, y_pred, dataset):
    disp = ConfusionMatrixDisplay.from_predictions(
        y_true,
        np.argmax(y_pred, axis=1),
        display_labels=["Not Disaster","Disaster"],
        cmap=plt.cm.Blues
    )

    tn, fp, fn, tp = confusion_matrix(y_true, np.argmax(y_pred, axis=1)).ravel()
    f1_score = tp / (tp+((fn+fp)/2))

    disp.ax_.set_title("Confusion Matrix on " + dataset + " Dataset -- F1 Score: " + str(f1_score.round(2)))
    plt.show()

print("\n=== モデル評価 ===")
y_pred_train = classifier.predict(X_train)
displayConfusionMatrix(y_train, y_pred_train, "Training")

y_pred_val = classifier.predict(X_val)
displayConfusionMatrix(y_val, y_pred_val, "Validation")


# テストデータで予測＆提出ファイル作成

In [ ]:
sample_submission = pd.read_csv("/kaggle/input/nlp-getting-started/sample_submission.csv")

sample_submission["target"] = np.argmax(classifier.predict(X_test), axis=1)
sample_submission.to_csv("submission.csv", index=False)
print("提出用ファイルは保存されました: submission.csv")
